# Notebook 26 — Volume, order-flow proxies and market breadth

**Research question:** Can ticker volume and contemporaneous market participation improve our seven-quantile classifier and class-1 entry quality? 3 chronological folds, 4 feature families, 6 models = **72 fits**. No historical holdout access or prospective ledger writes.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Open from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.volume_regime_research import run_volume_regime_research,CONFIGS
print('Project:',ROOT)

## 1. Research protocol

Fixed target `q=7`, mapping `[2,2,0,0,1,1,1]`. All transforms refit chronologically. We test balanced and unweighted controls explicitly; the final historical holdout is not evaluated.

In [ ]:
N_FOLDS=3
MAX_TRAIN_ROWS=60000
TREE_TRAIN_ROWS=40000
THRESHOLDS=(.45,.50,.55,.60)
COST_BPS=10
OUTPUT=ROOT/'artifacts'/'volume_regime'/'1d'
print('Model fits:',N_FOLDS*4*len(CONFIGS),'threshold diagnostics:',N_FOLDS*4*len(CONFIGS)*len(THRESHOLDS))

## 2. Load frozen master data

Load the master DataFrame, **not** the full four-item tuple. The research module internally reserves the historical holdout.

In [ ]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((x for x in paths if all((x/y).exists() for y in required)),None)
if source is None: raise FileNotFoundError('Master table artifacts not found; run Notebook 03 first')
master,latest,bundle,manifest=load_master(source)
print('Master rows:',len(master),'symbols:',master.symbol.nunique(),'last date:',master.bar_timestamp.max())
print('Volume missing:',round(master.volume.isna().mean(),4))

## 3. Run extensive three-fold experiment

This may take a long time. All 72 fits are intentional. We do not select a threshold from outer-fold results and then claim its performance is unbiased.

In [ ]:
results,leaderboard,diagnostics,cutoff=run_volume_regime_research(master,n_folds=N_FOLDS,
    max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,thresholds=THRESHOLDS,
    cost_bps=COST_BPS,output_dir=OUTPUT)
print('Historical development cutoff:',cutoff)
print('Rows:',len(results))
display(diagnostics)

## 4. Paired feature-family and model leaderboard

Accuracy is compared with the correct ~42% majority baseline. Entry precision is compared with the actual Up prevalence.

In [ ]:
display(leaderboard[['model','feature_set','mean_accuracy','mean_baseline','accuracy_lift','mean_auc','mean_entry_precision','mean_entry_coverage','mean_net_return','mean_fit_seconds']].head(24))
print('No model promoted: these are exploratory development folds.')

## 5. Which features add signal?

Compare paired fold performance with the same model, not different model configurations. Report accuracy *and* Up-entry behavior.

In [ ]:
ref=results[results.entry_threshold.eq(.50)]
pivot=ref.pivot_table(index=['fold','model'],columns='feature_set',values='accuracy')
pivot['volume_minus_price']=pivot['price_plus_volume']-pivot['price_only']
pivot['flow_minus_price']=pivot['price_plus_flow']-pivot['price_only']
display(pivot.sort_values('volume_minus_price',ascending=False))
print('Average paired volume lift:',pivot.volume_minus_price.mean())

## 6. Entry threshold diagnostics — descriptive only

Higher precision with vanishing coverage is not a robust entry strategy. Thresholds are **not** selected from these evaluation folds.

In [ ]:
display(results.groupby(['model','feature_set','entry_threshold'],as_index=False).agg(
    precision=('entry_precision','mean'),coverage=('entry_coverage','mean'),
    net_return=('mean_entry_net_return','mean'),entries=('entry_count','sum')).sort_values(['precision','coverage'],ascending=False).head(35))

## 7. Market conditions and economic target diagnostics

Inspect whether Up-label returns clear costs; a Q5 Up outcome can be unprofitable. A new economic target must be specified and evaluated in a future experiment, not selected from these test results.

In [ ]:
display(diagnostics[['fold','up_base','median_up_return','fraction_up_return_above_cost','volume_missing_rate']])
print('Data artifacts:',OUTPUT)

## 8. Decision rubric

**Advance to a new prospective shadow experiment only if** improvements are positive across all three folds, class-1 precision beats Up prevalence with useful coverage, and net return remains positive under reasonable costs. Otherwise shift to point-in-time sector membership, fundamentals, and news sentiment. Never treat this exploratory leaderboard as proof of trading profitability.